# 07 · Registrar a execução

Última tarefa do job. Ela roda **sempre**, com sucesso ou falha nas anteriores, porque usa a dependência **Run if: All done** (aula 16). Grava uma linha em `growdev.gold.log_execucoes` com o resultado da validação, formando um histórico simples das execuções.

In [ ]:
dbutils.widgets.text("catalogo", "growdev", "Catálogo")
dbutils.widgets.text("job_run_id", "manual", "Id da execução do job")

CATALOGO = dbutils.widgets.get("catalogo")
JOB_RUN_ID = dbutils.widgets.get("job_run_id")

In [ ]:
from pyspark.sql import functions as F

qualidade_ok = dbutils.jobs.taskValues.get(
    taskKey="validar_qualidade", key="qualidade_ok", default=None, debugValue=None
)

log = spark.createDataFrame(
    [(JOB_RUN_ID, str(qualidade_ok))],
    "job_run_id string, qualidade_ok string",
).withColumn("registrado_em", F.current_timestamp())

log.write.mode("append").saveAsTable(f"{CATALOGO}.gold.log_execucoes")
display(spark.table(f"{CATALOGO}.gold.log_execucoes").orderBy(F.desc("registrado_em")).limit(10))